In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import random
import csv
import json

import numpy as np
import config

import yaml

load_dotenv(find_dotenv())
engine = create_engine(f'postgresql://{config.db_username}:{config.db_password}@{config.db_host}:{config.db_port}/{config.db_name}')
connection = engine.connect()

# Utils for migration

In [2]:
import time
import re

# Function to remove comments from SQL file
def remove_comments(sql_content):
    # Remove single-line comments (--) and multi-line comments (/* ... */)
    sql_content = re.sub(r'--.*', '', sql_content)  # Remove single-line comments
    sql_content = re.sub(r'/\*.*?\*/', '', sql_content, flags=re.DOTALL)  # Remove block comments
    return sql_content


def initialize_db():
    with engine.connect() as connection:
        try:
            with connection.begin():
                connection.execute(text('DROP SCHEMA IF EXISTS tmp_migration_db CASCADE'))
                connection.execute(text('CREATE SCHEMA tmp_migration_db'))
                connection.execute(text('SET search_path TO tmp_migration_db, public'))
            # print("initialized to ",connection.execute(text('SHOW search_path')).fetchone())
        except Exception as e:
            print(f"Error executing initialization")


def simulate_migration(migration_file_name, num_sims):
    log_entries = []

    # Load and clean the SQL file
    with open(f'../migrations/{migration_file_name}.sql', 'r') as file:
        sql_content = file.read()

    # Remove comments and split into individual queries
    cleaned_sql = remove_comments(sql_content)
    queries = [query.strip() for query in cleaned_sql.split(';') if query.strip()]

    for i in range(num_sims):
        print(f'Run {i+1}/{num_sims}')
        initialize_db()

        # Track the execution time
        start_time = time.time()

        # Execute each query in the file
        with engine.connect() as connection:
            with connection.begin():
                for query in queries:
                    try:
                        connection.execute(text(query))
                        # print(query)
                        # log_entry = rows[0][0]
                        # log_entry['QueryType'] = (chosen_index + 1)
                        # log_entry['NumRows'] = rows[0][0]["Plan"]["Actual Rows"]
                        # log_entry['FullQuery'] = pre_query
                        # log_entry['QueryParams'] = query_params

                        # log_entries.append(log_entry)
                    except Exception as e:
                        print(f"Error executing query: {query}\nError: {e}")

        # Calculate the total execution time
        end_time = time.time()
        execution_time = end_time - start_time

        log_entry = dict()
        log_entry['Sim'] = i
        log_entry['Time'] = execution_time
        log_entries.append(log_entry)

    log_df = pd.DataFrame(log_entries)
    return log_df

# Execution

In [7]:
scale = '2'
mig_conf = []
# mig_conf.append({ 'migration_file_name': '2003-Optimized' })
mig_conf.append({ 'migration_file_name': '2013-Optimized' })
mig_conf.append({ 'migration_file_name': '2023-Optimized' })

num_sims = 5

outdir = (f"../results")
if not os.path.exists(outdir):
    os.mkdir(outdir)

for i in range(len(mig_conf)):
    print(f'** RUNNING {scale}x/{mig_conf[i]['migration_file_name']} ({num_sims} migrations) **')
    log_df = simulate_migration(f'{scale}x/{mig_conf[i]["migration_file_name"]}', num_sims)
    log_df.to_csv(f"{outdir}/results_mig_{scale}x_{mig_conf[i]['migration_file_name']}.csv",index=False)

** RUNNING 2x/2013-Optimized (5 migrations) **
Run 1/5
Run 2/5
Run 3/5
Run 4/5
Run 5/5
** RUNNING 2x/2023-Optimized (5 migrations) **
Run 1/5
Run 2/5
Run 3/5
Run 4/5
Run 5/5
